# U09_P — Protein Prediction — Solutions
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 9:** Predictive Methods using Protein Sequence
**Type:** Practice Solutions notebook · **Time:** 50 min · **Difficulty:** ★★☆
**Goes with:** Lectures L23–L25, Practicals S10–S11
**Before this:** U09_C1, U09_C2

In [1]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U09_P"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

Python 3.11.18 | Biopython 1.88


### Ex 9.1 ★☆☆ — Compute molecular weight and isoelectric point for human HBB
**Biology:** Beta-globin is the 147-aa subunit of adult hemoglobin.  
**Task:** Load `hbb_human_protein.fasta`. Use `ProteinAnalysis` to calculate molecular weight `hbb_mw` and isoelectric point `hbb_pi` as floats rounded to 2 decimal places.  
**Expected:** `hbb_mw` ~15998 Da, `hbb_pi` ~6.74.

In [2]:
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

hbb_seq = str(SeqIO.read(DATA_DIR / "sequences" / "hbb_human_protein.fasta", "fasta").seq)
pa = ProteinAnalysis(hbb_seq)
hbb_mw = round(pa.molecular_weight(), 2)
hbb_pi = round(pa.isoelectric_point(), 2)

In [3]:
# Check — do not edit
assert 15900 < hbb_mw < 16100, f"Expected MW ~16000 Da, got {hbb_mw}"
assert 6.5 < hbb_pi < 7.0, f"Expected pI ~6.74, got {hbb_pi}"
print("Ex 9.1 passed")

Ex 9.1 passed


### Ex 9.2 ★☆☆ — Measure the pI shift caused by the sickle-cell mutation
**Biology:** Replacing negatively charged Glutamic acid with non-polar Valine shifts the isoelectric point more basic.  
**Task:** Load `hbs_human_protein.fasta`. Calculate `hbs_pi`. Store the difference `hbs_pi - hbb_pi` in `pi_shift`.  
**Expected:** `pi_shift` as a positive float between 0.3 and 0.5.

In [4]:
hbs_seq = str(SeqIO.read(DATA_DIR / "sequences" / "hbs_human_protein.fasta", "fasta").seq)
pa_s = ProteinAnalysis(hbs_seq)
hbs_pi = round(pa_s.isoelectric_point(), 2)
pi_shift = round(hbs_pi - hbb_pi, 2)

In [5]:
# Check — do not edit
assert pi_shift > 0, "HbS pI must be higher than HbA pI due to loss of negative glutamate"
assert 0.3 <= pi_shift <= 0.5, f"Expected pI shift ~0.39, got {pi_shift}"
print("Ex 9.2 passed")

Ex 9.2 passed


### Ex 9.3 ★☆☆ — Calculate GRAVY hydropathy score for human insulin
**Biology:** The GRAVY score reflects the hydrophobic/hydrophilic balance of preproinsulin.  
**Task:** Load `ins_human_protein.fasta`. Use `ProteinAnalysis` to calculate its GRAVY hydropathy score in `ins_gravy` rounded to 3 decimal places.  
**Expected:** `ins_gravy` as float.

In [6]:
ins_seq = str(SeqIO.read(DATA_DIR / "sequences" / "ins_human_protein.fasta", "fasta").seq)
pa_ins = ProteinAnalysis(ins_seq)
ins_gravy = round(pa_ins.gravy(), 3)

In [7]:
# Check — do not edit
assert isinstance(ins_gravy, float)
assert -0.5 <= ins_gravy <= 0.5, f"Unexpected GRAVY value: {ins_gravy}"
print("Ex 9.3 passed")

Ex 9.3 passed


### Ex 9.4 ★★☆ — Translate a PROSITE pattern into a Python regular expression
**Biology:** PROSITE syntax uses brackets `[ABC]`, excludes `{D}`, and wildcard counts `x(N)`.  
**Task:** Translate the PROSITE pattern `'[LIVM]-x(2)-[DE]'` into a Python regex string stored in `my_regex`. Verify it matches test peptide `'LAAEE'`.  
**Expected:** `my_regex` equals `'[LIVM].{2}[DE]'`.

In [8]:
my_regex = "[LIVM].{2}[DE]"

In [9]:
# Check — do not edit
import re
assert re.search(my_regex, "LAAEE") is not None, "Regex must match test peptide"
print("Ex 9.4 passed")

Ex 9.4 passed


### Ex 9.5 ★★☆ — Scan human beta-globin for the distal histidine motif
**Biology:** The haem pocket in hemoglobin is stabilized by the distal Histidine (His63 in HBB).  
**Task:** Using regex pattern `r'[LIVMF].{2}H.{2}[KR]'`, search `hbb_seq`. Extract the matching peptide substring into `matched_subseq`.  
**Expected:** `matched_subseq` of length 7 containing 'H'.

In [10]:
import re

motif_regex = r"[LIVMF].{2}H.{2}[KR]"
match = re.search(motif_regex, hbb_seq)
matched_subseq = match.group() if match else ""

In [11]:
# Check — do not edit
assert len(matched_subseq) == 7, f"Expected 7 aa motif match, got {len(matched_subseq)}"
assert "H" in matched_subseq, "Motif must contain coordinating Histidine"
print("Ex 9.5 passed")

Ex 9.5 passed


### Ex 9.6 ★★☆ — Compute Q3 accuracy for PSIPRED against DSSP ground truth
**Biology:** Benchmarking secondary structure prediction accuracy against experimental X-ray structures.  
**Task:** Given two 3-state strings `pred_str = 'CCHHHHHHCCCCCCCC'` and `true_str = 'CCHHHHHHCCCCHHHH'`, write a function `calc_q3(pred, true)` returning accuracy percentage rounded to 1 decimal place. Compute for `pred_str` and `true_str` in `test_q3`.  
**Expected:** `test_q3` = 75.0% (12/16 correct).

In [12]:
def calc_q3(pred, true):
    n = min(len(pred), len(true))
    matches = sum(1 for i in range(n) if pred[i] == true[i])
    return round((matches / n) * 100, 1)

pred_str = "CCHHHHHHCCCCCCCC"
true_str = "CCHHHHHHCCCCHHHH"
test_q3 = calc_q3(pred_str, true_str)

In [13]:
# Check — do not edit
assert test_q3 == 75.0, f"Expected 75.0%, got {test_q3}"
print("Ex 9.6 passed")

Ex 9.6 passed


### Ex 9.7 ★★★ — Compare PSIPRED vs GOR IV accuracy on full HBB sequence
**Biology:** Evolutionary profile neural networks (PSIPRED) outperform single-sequence propensity statistics (GOR IV).  
**Task:** Load `psipred_hbb_secondary_structure.txt`, `gor4_hbb_secondary_structure.txt`, and `dssp_hbb_1a3n.txt` from `data/expected/`. Calculate `q3_psipred` and `q3_gor4`. Verify both exceed 70.0%.  
**Expected:** Both `q3_psipred` and `q3_gor4` > 70.0.

In [14]:
def get_string(p):
    with open(p) as f:
        for line in f:
            line = line.strip()
            if line and set(line).issubset(set("HECT ")):
                return line.replace(" ", "")
    return ""

exp_dir = DATA_DIR / "expected"
psi_s = get_string(exp_dir / "psipred_hbb_secondary_structure.txt")
gor_s = get_string(exp_dir / "gor4_hbb_secondary_structure.txt")
dssp_s = get_string(exp_dir / "dssp_hbb_1a3n.txt")
dssp_3 = "".join("H" if c in "HG" else ("E" if c == "E" else "C") for c in dssp_s)

q3_psipred = calc_q3(psi_s, dssp_3)
q3_gor4 = calc_q3(gor_s, dssp_3)

In [15]:
# Check — do not edit
assert q3_psipred > 70.0, f"Expected PSIPRED Q3 > 70%, got {q3_psipred}"
assert q3_gor4 > 70.0, f"Expected GOR4 Q3 > 70%, got {q3_gor4}"
print("Ex 9.7 passed")

Ex 9.7 passed


## 5. Reflection Questions
1. Why does replacing a charged amino acid with a hydrophobic one on a protein surface promote pathological fiber assembly in sickle-cell anemia?
2. What are the three states in $Q_3$ accuracy scoring, and why is $Q_3$ considered a better evaluation standard than per-residue root mean square deviation (RMSD) for primary sequence prediction?